# 07 · Riesgo mensual de roya y muerte descendente y alertas tempranas

Calcula, mes a mes, la favorabilidad climática para roya y muerte descendente (*Phoma* spp.) sobre el área sembrada de café del Cauca, y emite un nivel de alerta por municipio.

**Cómo se arma cada mes**

| Variable | Fuente | Cómo |
|---|---|---|
| Lluvia del mes | CHIRPS diario | suma del mes |
| Temperatura media del mes | TerraClimate + ERA5-Land | climatología 1995–2024 corregida por altitud + anomalía mensual de ERA5-Land |
| Humedad relativa del mes | ERA5-Land mensual | punto de rocío y temperatura (Magnus) |
| Amplitud térmica | TerraClimate | climatología del mismo mes del año |
| Altitud | SRTM | fija |

**Alertas por municipio**: porcentaje del área de café en riesgo alto o muy alto en el último mes disponible, y anomalía del índice frente al promedio del mismo mes en los años de referencia.

ERA5-Land se publica con unos 3 meses de rezago, así que el "último mes" suele ser de hace 2 o 3 meses. Los índices son de favorabilidad climática relativa: no predicen incidencia hasta que se calibren con datos de campo.

## 1. Conexión a Earth Engine

In [ ]:
import ee, geemap
from ipyleaflet import basemaps

PROYECTO_GEE = 'TU_PROYECTO_GEE'   # <-- tu ID de proyecto de Google Cloud, entre comillas

# Autenticación con enlace y código (evita el error "credential propagation was unsuccessful").
# Al generar el token, NO marques la opción de solo lectura: exportar necesita permisos completos.
ee.Authenticate(auth_mode='notebook')
ee.Initialize(project=PROYECTO_GEE)

## 2. Configuración

Los criterios usan los mismos umbrales del notebook 06, salvo la lluvia, que aquí es **mensual** (mm/mes). Si cambias criterios en el 06, cámbialos también aquí.

In [ ]:
PERIODO_CLIMA = ('1995-01-01', '2025-01-01')   # referencia para climatologías
GRADIENTE_TERMICO = -0.006                     # °C por metro

FUENTE_AREA_CAFE = 'fdp'      # 'fdp' o 'asset' (ver notebook 06)
ANIO_CAFE = 2024
UMBRAL_PROB_CAFE = 0.5
ASSET_AREA_CAFE = 'projects/TU_PROYECTO_GEE/assets/area_cafe_cauca'

N_MESES = 24                                   # meses de la serie, hasta el último disponible
BASE_ANOMALIA = (2015, 2024)                   # años para el promedio de cada mes (anomalías)
ESCALA_SERIE = 250                             # m, para la serie departamental
ESCALA_MUNICIPIO = 100                         # m, para las alertas por municipio

# Niveles de alerta: % del área de café del municipio en riesgo alto o muy alto
UMBRAL_ATENCION = 30
UMBRAL_ALERTA = 50
AREA_MINIMA_HA = 100           # municipios con menos café no se evalúan

RIESGO_ROYA_MES = {
    'temperatura': {'trapecio': [15, 21, 25, 28], 'peso': 0.45},  # °C, media del mes
    'altitud':     {'baja':     [1300, 2000],     'peso': 0.30},  # m
    'lluvia':      {'sube':     [125, 210],       'peso': 0.15},  # mm/mes (1.500-2.500 / 12)
    'humedad':     {'sube':     [78, 92],         'peso': 0.10},  # %
}

RIESGO_PHOMA_MES = {
    'altitud':     {'sube': [1400, 1900], 'peso': 0.35},  # m
    'temperatura': {'baja': [17, 21],     'peso': 0.25},  # °C
    'amplitud':    {'sube': [9.5, 11.5],  'peso': 0.20},  # °C día-noche
    'lluvia':      {'sube': [150, 265],   'peso': 0.20},  # mm/mes (1.800-3.200 / 12)
}

for nombre, crit in [('roya', RIESGO_ROYA_MES), ('phoma', RIESGO_PHOMA_MES)]:
    assert abs(sum(v['peso'] for v in crit.values()) - 1) < 1e-6, f'Pesos de {nombre} no suman 1'

## 3. Área de estudio y área sembrada

In [ ]:
cauca = (ee.FeatureCollection('FAO/GAUL/2015/level1')
         .filter(ee.Filter.eq('ADM0_NAME', 'Colombia'))
         .filter(ee.Filter.eq('ADM1_NAME', 'Cauca')))   # nombre exacto: excluye Valle del Cauca
region = cauca.geometry()

municipios = (ee.FeatureCollection('FAO/GAUL/2015/level2')
              .filter(ee.Filter.eq('ADM0_NAME', 'Colombia'))
              .filter(ee.Filter.eq('ADM1_NAME', 'Cauca')))

print('Departamentos encontrados:', cauca.size().getInfo())   # debe ser 1
print('Municipios:', municipios.size().getInfo())            # GAUL 2015 puede no dar 42

In [ ]:
def area_sembrada_cafe():
    """Máscara (1 = café) del área sembrada, según FUENTE_AREA_CAFE."""
    if FUENTE_AREA_CAFE == 'fdp':
        prob = (ee.ImageCollection('projects/forestdatapartnership/assets/coffee/model_2026a')
                .filterDate(f'{ANIO_CAFE}-01-01', f'{ANIO_CAFE + 1}-01-01')
                .mosaic().select('probability'))
        return prob.gte(UMBRAL_PROB_CAFE).selfMask().rename('cafe').clip(region)
    if FUENTE_AREA_CAFE == 'asset':
        poligonos = ee.FeatureCollection(ASSET_AREA_CAFE)
        return ee.Image().byte().paint(poligonos, 1).selfMask().rename('cafe').clip(region)
    raise ValueError("FUENTE_AREA_CAFE debe ser 'fdp' o 'asset' en este paso")

def hectareas(mascara, escala=30):
    res = (ee.Image.pixelArea().divide(10000).updateMask(mascara)
           .reduceRegion(ee.Reducer.sum(), region, escala, maxPixels=1e11, tileScale=8))
    return ee.Number(res.get('area'))

cafe = area_sembrada_cafe()
print(f'Área de café ({FUENTE_AREA_CAFE}): {hectareas(cafe).getInfo():,.0f} ha')

## 4. Funciones

In [ ]:
def puntaje(img, regla):
    """Convierte una capa a un puntaje de 0 a 1 según la regla del criterio."""
    if 'trapecio' in regla:
        a, b, c, d = regla['trapecio']
        sube = img.subtract(a).divide(b - a)
        baja = ee.Image.constant(d).subtract(img).divide(d - c)
        return sube.min(baja).clamp(0, 1)
    if 'sube' in regla:
        a, b = regla['sube']
        return img.subtract(a).divide(b - a).clamp(0, 1)
    if 'baja' in regla:
        a, b = regla['baja']
        return ee.Image.constant(b).subtract(img).divide(b - a).clamp(0, 1)
    raise ValueError(f'Regla no reconocida: {regla}')

def pesos_base(criterios):
    return {v: r['peso'] for v, r in criterios.items()}

def indice(criterios, nombre, img=None, pesos=None):
    """Suma ponderada de los puntajes (0 a 1). img: imagen con las variables; pesos: opcional."""
    img = capas if img is None else img
    pesos = pesos_base(criterios) if pesos is None else pesos
    total = ee.Image.constant(0)
    for variable, regla in criterios.items():
        total = total.add(puntaje(img.select(variable), regla).multiply(pesos[variable]))
    return total.rename(nombre).clip(region)

def clasificar(img):
    """Clases 1 a 5 con cortes en 0,2 / 0,4 / 0,6 / 0,8."""
    return (img.gte(0.2).add(img.gte(0.4)).add(img.gte(0.6)).add(img.gte(0.8))
            .add(1).rename('clase').updateMask(img.mask()))

def humedad_relativa(t, td):
    """Humedad relativa (%) con la aproximación de Magnus; t y td en °C."""
    pv = lambda x: x.multiply(17.625).divide(x.add(243.04)).exp()
    return pv(td).divide(pv(t)).multiply(100)

def hex_a_rgba(h):
    h = h.lstrip('#')
    return (int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16), 255)

PALETA_APT = ['#d7191c', '#fdae61', '#ffffbf', '#a6d96a', '#1a9641']
ETIQ_APT = ['No apto', 'Marginal', 'Moderado', 'Apto', 'Muy apto']
PALETA_RIESGO = ['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#d7191c']
ETIQ_RIESGO = ['Muy bajo', 'Bajo', 'Medio', 'Alto', 'Muy alto']
COLOR_FUERA = '#d9d9d9'

## 5. Capas fijas y climatologías

In [ ]:
# --- Relieve (SRTM, 30 m)
dem = ee.Image('USGS/SRTMGL1_003')
altitud = dem.rename('altitud')

# --- Climatología reciente de temperatura (TerraClimate, ~4,6 km) con corrección altitudinal
terra = (ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE')
         .filterDate(*PERIODO_CLIMA)
         .map(lambda i: i.resample('bilinear')))
terra_proj = ee.Image(ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE').first()).projection()

t_media_gruesa = terra.map(lambda i: i.select('tmmx').add(i.select('tmmn'))
                           .multiply(0.05)).mean()            # (tmmx + tmmn) / 2, escala 0,1
dem_grueso = (dem.reduceResolution(ee.Reducer.mean(), maxPixels=65535)
              .reproject(crs=terra_proj).resample('bilinear'))
t_media_fina = t_media_gruesa.add(dem.subtract(dem_grueso).multiply(GRADIENTE_TERMICO))

def amplitud_de(col):
    return col.map(lambda i: i.select('tmmx').subtract(i.select('tmmn')).multiply(0.1)).mean()

# Amplitud térmica típica de cada mes del año (1 a 12)
AMPLITUD_MES = ee.ImageCollection.fromImages([
    amplitud_de(terra.filter(ee.Filter.calendarRange(m, m, 'month'))).rename('amplitud').set('mes', m)
    for m in range(1, 13)])

# ERA5-Land mensual: temperatura media de referencia, para calcular anomalías
ERA5 = (ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
        .map(lambda img: img.resample('bilinear')))
T_ERA_CLIM = (ERA5.filterDate(*PERIODO_CLIMA).select('temperature_2m').mean()
              .subtract(273.15))

CHIRPS = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')

def capas_mes(anio, mes):
    """Imagen con las variables del mes indicado."""
    inicio = ee.Date.fromYMD(anio, mes, 1)
    fin = inicio.advance(1, 'month')
    era_m = ERA5.filterDate(inicio, fin).first()
    t_era = era_m.select('temperature_2m').subtract(273.15)
    td_era = era_m.select('dewpoint_temperature_2m').subtract(273.15)
    temperatura = t_media_fina.add(t_era.subtract(T_ERA_CLIM)).rename('temperatura')
    humedad = humedad_relativa(t_era, td_era).rename('humedad')
    lluvia = CHIRPS.filterDate(inicio, fin).sum().rename('lluvia')
    amplitud = AMPLITUD_MES.filter(ee.Filter.eq('mes', mes)).first()
    return (ee.Image.cat([altitud, temperatura, humedad, lluvia, amplitud]).clip(region)
            .set('system:time_start', inicio.millis()))

def riesgo_mes(anio, mes):
    img = capas_mes(anio, mes)
    roya = indice(RIESGO_ROYA_MES, 'roya', img=img).updateMask(cafe)
    phoma = indice(RIESGO_PHOMA_MES, 'phoma', img=img).updateMask(cafe)
    return roya.addBands(phoma)

## 6. Último mes disponible

In [ ]:
from datetime import date, timedelta

def ultimo_mes_completo(ultimo_dia):
    """(año, mes) del último mes con todos sus días disponibles."""
    if (ultimo_dia + timedelta(days=1)).month != ultimo_dia.month:
        return ultimo_dia.year, ultimo_dia.month
    primero = ultimo_dia.replace(day=1) - timedelta(days=1)
    return primero.year, primero.month

def meses_hacia_atras(anio, mes, n):
    salida = []
    for _ in range(n):
        salida.append((anio, mes))
        anio, mes = (anio, mes - 1) if mes > 1 else (anio - 1, 12)
    return salida[::-1]

ms_era = ERA5.aggregate_max('system:time_start').getInfo()
ms_chirps = CHIRPS.aggregate_max('system:time_start').getInfo()
d_era = date.fromtimestamp(ms_era / 1000)
d_chirps = date.fromtimestamp(ms_chirps / 1000)

fin_era = (d_era.year, d_era.month)                 # ERA5 mensual: cada imagen es un mes completo
fin_chirps = ultimo_mes_completo(d_chirps)
ULTIMO = min(fin_era, fin_chirps)
MESES = meses_hacia_atras(*ULTIMO, N_MESES)

print('ERA5-Land hasta:', fin_era, '| CHIRPS hasta:', fin_chirps)
print('Último mes analizado:', ULTIMO, '| Serie desde:', MESES[0])

## 7. Serie mensual del riesgo en el área sembrada

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def resumen_mes(anio, mes):
    r = riesgo_mes(anio, mes)
    altos = r.gte(0.6).rename(['roya_alto', 'phoma_alto'])
    valores = r.addBands(altos).reduceRegion(
        ee.Reducer.mean(), region, ESCALA_SERIE, maxPixels=1e10, tileScale=8)
    return ee.Feature(None, valores).set({'anio': anio, 'mes': mes})

serie_fc = ee.FeatureCollection([resumen_mes(a, m) for a, m in MESES]).getInfo()
serie = pd.DataFrame([f['properties'] for f in serie_fc['features']])
serie['fecha'] = pd.to_datetime(dict(year=serie['anio'], month=serie['mes'], day=1))
serie = serie.set_index('fecha').sort_index()
serie['% área roya alto+'] = (100 * serie['roya_alto']).round(1)
serie['% área Phoma alto+'] = (100 * serie['phoma_alto']).round(1)
display(serie[['roya', 'phoma', '% área roya alto+', '% área Phoma alto+']].round(3))

fig, ejes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ejes[0].plot(serie.index, serie['roya'], marker='o', color='#d7191c', label='Roya')
ejes[0].plot(serie.index, serie['phoma'], marker='o', color='#2c7bb6', label='Phoma')
ejes[0].set_ylabel('Índice medio (0 a 1)'); ejes[0].set_ylim(0, 1); ejes[0].legend()
ejes[1].plot(serie.index, serie['% área roya alto+'], marker='o', color='#d7191c', label='Roya')
ejes[1].plot(serie.index, serie['% área Phoma alto+'], marker='o', color='#2c7bb6', label='Phoma')
ejes[1].axhline(UMBRAL_ATENCION, color='#fdae61', ls='--', lw=1)
ejes[1].axhline(UMBRAL_ALERTA, color='#d7191c', ls='--', lw=1)
ejes[1].set_ylabel('% del área de café en riesgo alto o muy alto'); ejes[1].set_ylim(0, 100)
for e in ejes: e.grid(alpha=0.3)
plt.suptitle('Riesgo mensual sobre el área sembrada de café, Cauca')
plt.tight_layout(); plt.show()

## 8. Mapas del último mes

In [ ]:
riesgo_ultimo = riesgo_mes(*ULTIMO)
sin_cafe = cafe.unmask(0).eq(0).selfMask().clip(region)

def mapa_mes(banda, titulo):
    m = geemap.Map(basemap=basemaps.Esri.WorldTopoMap)
    m.centerObject(cauca, 8)
    m.addLayer(sin_cafe, {'palette': [COLOR_FUERA]}, 'Sin café', True, 0.75)
    m.addLayer(clasificar(riesgo_ultimo.select(banda)),
               {'min': 1, 'max': 5, 'palette': PALETA_RIESGO}, titulo)
    m.addLayer(municipios.style(color='#555555', fillColor='00000000', width=1), {}, 'Municipios')
    m.addLayer(cauca.style(color='black', fillColor='00000000', width=2), {}, 'Cauca')
    leyenda = dict(zip(ETIQ_RIESGO, PALETA_RIESGO)); leyenda['Sin café'] = COLOR_FUERA
    m.add_legend(title=titulo, legend_dict=leyenda)
    return m

TXT_MES = f'{ULTIMO[0]}-{ULTIMO[1]:02d}'

In [ ]:
mapa_mes('roya', f'Riesgo de roya {TXT_MES}')

In [ ]:
mapa_mes('phoma', f'Riesgo de Phoma {TXT_MES}')

## 9. Alertas por municipio

- **% alto+**: porcentaje del área de café del municipio con índice ≥ 0,6 (riesgo alto o muy alto) en el último mes.
- **Anomalía**: índice medio del último mes menos el promedio del mismo mes en `BASE_ANOMALIA`. Positiva = condiciones más favorables a la enfermedad que lo normal para esa época.
- **Nivel**: *Alerta* si % alto+ ≥ `UMBRAL_ALERTA`; *Atención* si ≥ `UMBRAL_ATENCION`; si no, *Normal*.

In [ ]:
anio_u, mes_u = ULTIMO
clim_mes = ee.ImageCollection.fromImages(
    [riesgo_mes(a, mes_u) for a in range(BASE_ANOMALIA[0], BASE_ANOMALIA[1] + 1)]).mean()

area = ee.Image.pixelArea().divide(10000).updateMask(cafe)
img = ee.Image.cat([
    area.rename('ha_cafe'),
    area.multiply(riesgo_ultimo.select('roya').gte(0.6)).rename('ha_roya_alto'),
    area.multiply(riesgo_ultimo.select('phoma').gte(0.6)).rename('ha_phoma_alto'),
    area.multiply(riesgo_ultimo.select('roya')).rename('roya_x_ha'),
    area.multiply(riesgo_ultimo.select('phoma')).rename('phoma_x_ha'),
    area.multiply(clim_mes.select('roya')).rename('roya_clim_x_ha'),
    area.multiply(clim_mes.select('phoma')).rename('phoma_clim_x_ha'),
]).unmask(0)

por_mpio = img.reduceRegions(collection=municipios.select(['ADM2_NAME']),
                             reducer=ee.Reducer.sum(), scale=ESCALA_MUNICIPIO,
                             tileScale=8).getInfo()

def nivel(pct):
    if pct >= UMBRAL_ALERTA: return 'Alerta'
    if pct >= UMBRAL_ATENCION: return 'Atención'
    return 'Normal'

filas = []
for f in por_mpio['features']:
    p = f['properties']; ha = p.get('ha_cafe') or 0
    if ha < AREA_MINIMA_HA:
        continue
    fila = {'municipio': p['ADM2_NAME'], 'ha café': round(ha)}
    for enf in ['roya', 'phoma']:
        pct = 100 * p[f'ha_{enf}_alto'] / ha
        fila[f'{enf} % alto+'] = round(pct, 1)
        fila[f'{enf} anomalía'] = round((p[f'{enf}_x_ha'] - p[f'{enf}_clim_x_ha']) / ha, 3)
        fila[f'{enf} nivel'] = nivel(pct)
    filas.append(fila)

alertas = pd.DataFrame(filas).sort_values(['roya % alto+', 'phoma % alto+'], ascending=False)
print(f'Alertas para {TXT_MES} ({len(alertas)} municipios con al menos {AREA_MINIMA_HA} ha de café)')
display(alertas.reset_index(drop=True))

In [ ]:
COLORES_NIVEL = {'Normal': '#1a9641', 'Atención': '#fdae61', 'Alerta': '#d7191c'}

def mapa_alertas(enf, titulo):
    m = geemap.Map(basemap=basemaps.Esri.WorldTopoMap)
    m.centerObject(cauca, 8)
    for nv, color in COLORES_NIVEL.items():
        nombres = alertas.loc[alertas[f'{enf} nivel'] == nv, 'municipio'].tolist()
        if nombres:
            fc = municipios.filter(ee.Filter.inList('ADM2_NAME', nombres))
            m.addLayer(fc.style(color='#555555', fillColor=color.lstrip('#') + 'b0', width=1), {}, nv)
    m.addLayer(municipios.style(color='#555555', fillColor='00000000', width=1), {}, 'Municipios')
    m.addLayer(cauca.style(color='black', fillColor='00000000', width=2), {}, 'Cauca')
    leyenda = dict(COLORES_NIVEL); leyenda['Sin evaluar'] = '#ffffff'
    m.add_legend(title=titulo, legend_dict=leyenda)
    return m

In [ ]:
mapa_alertas('roya', f'Alerta de roya {TXT_MES}')

In [ ]:
mapa_alertas('phoma', f'Alerta de Phoma {TXT_MES}')

## 10. Descargar resultados (CSV)

In [ ]:
from google.colab import files

serie.to_csv(f'serie_riesgo_mensual_{TXT_MES}.csv')
alertas.to_csv(f'alertas_municipios_{TXT_MES}.csv', index=False)
files.download(f'serie_riesgo_mensual_{TXT_MES}.csv')
files.download(f'alertas_municipios_{TXT_MES}.csv')